In [33]:
import os 

In [34]:
os.chdir('D:\PredictBot-Score-MLOps')

In [35]:
%pwd

'D:\\PredictBot-Score-MLOps'

In [36]:
from src.predictor_bot_score.logger import logger
from src.predictor_bot_score.config.configuration import yaml_load , create_directories
from src.predictor_bot_score.constants import CONFIG_PATH
from sklearn.metrics import mean_absolute_error
from dataclasses import dataclass
from src.predictor_bot_score.utils.model_factory import get_model , get_fit_kwargs , get_mlflow_logger , MODEL_REGISTRY
from src.predictor_bot_score.utils.src_util_s3_ import *
import glob
from datetime import datetime
import mlflow
import shutil
import pandas as pd
import pickle
import json
import numpy as np
from pathlib import Path
import time
import gc

In [37]:
@dataclass(frozen=True)
class ModelEvaluationConfig:
    test_data_path: Path
    validation_data_path : Path
    model_dir: Path
    model_eval_dir: Path
    champion_path: Path
    model_eval_results: Path  # Changed from report_dir to match YAML
    features: list[str]
    target_column: str
    baseline_feature: str
    baseline_mae: float
    improvement_threshold: float
    champion_threshold: float
    spike_threshold: float
    mlflow_experiment: str
    mlflow_tracking_uri: str
    model_training: dict
    bucket_name           : str    # new
    models                : dict

In [38]:
class Config_manager:

    def __init__(self ,config = CONFIG_PATH):
        self.config = yaml_load(config)
        
        create_directories([self.config.artifacts_root])
    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:

        config = self.config.model_evaluation
        training_config = self.config.model_training.models

        create_directories([config.model_eval_dir, config.model_eval_results])
        
        return ModelEvaluationConfig(
        test_data_path        = Path(config.test_data_path),
        validation_data_path = Path(config.validation_data_path),
        model_dir             = Path(config.model_dir),
        model_eval_dir        = Path(config.model_eval_dir),
        champion_path         = Path(config.champion_path),
        model_eval_results    = Path(config.model_eval_results),
        features              = list(config.features),
        target_column         = config.target_column,
        baseline_feature      = config.baseline_feature,
        baseline_mae          = float(config.baseline_mae),
        improvement_threshold = float(config.improvement_threshold),
        champion_threshold    = float(config.champion_threshold),
        spike_threshold       = float(config.spike_threshold),
        mlflow_experiment     = config.mlflow.experiment_name,
        mlflow_tracking_uri   = config.mlflow.tracking_uri,
        model_training        = dict(training_config),
        bucket_name           = self.config.s3_config.bucket_name,          # new
        models                = dict(training_config)
    )

In [ ]:
class Model_evalulation:

    def __init__(self,config : ModelEvaluationConfig):
        self.config = config
        self.s3 = s3_login()
        self.BUCKET_NAME = self.config.bucket_name
        self.test_df = self._read_data(prefix= "test")
        self.val_df  = self._read_data(prefix= "val")
        
        
        

    def _read_data(self,  prefix):
        try:
            keys = []
            paginator = self.s3.get_paginator('list_objects_v2')
            
            for page in paginator.paginate(Bucket=self.BUCKET_NAME, Prefix='feature_engineering'):
                for obj in page.get('Contents', []):
                    key = obj['Key']
                    if prefix in key and key.endswith('.csv') :
                        keys.append(key)

            if not keys:
                raise FileNotFoundError(
                    f"No {prefix} CSV found under s3://{self.BUCKET_NAME}/feature_engineering"
                )
            latest_key = sorted(keys)[-1]
            logger.info(f"Reading {prefix} data from: {latest_key}")

            obj = self.s3.get_object(Bucket=self.BUCKET_NAME, Key=latest_key)
            df  = pd.read_csv(io.BytesIO(obj['Body'].read()))
            logger.info(f"{prefix} rows: {len(df)}")
            return df
        
        except ClientError as e:
            logger.error(f"S3 error reading {prefix}: {e.response['Error']['Message']}")
            raise
        

    def _latest_models(self):
        try:
            latest_folder = glob.glob(os.path.join(self.config.model_dir ,"run__*" ))

            if not latest_folder:
                raise FileNotFoundError(f"No run folders found in {self.config.model_dir}")
            
            latest_model_file = max(latest_folder , key=os.path.getmtime)

            model_files = glob.glob(os.path.join(latest_model_file, "*.pkl"))
    
            return model_files          
        except FileNotFoundError as e:
                logger.error(f"Model file not found: {e}")
                raise

        except Exception as e:
            logger.error(f"Failed to load model: {str(e)}")
            raise       
    
    def _prepare_data(self ,df):

        try:
            X = df[self.config.features]
            y = df[self.config.target_column]

            return X ,y
        except Exception as e:
            raise
    
    def _smape(self, actual, predicted):
        try:
            return float(
                100 * np.mean(
                    2 * np.abs(predicted - actual) /
                    (np.abs(actual) + np.abs(predicted) + 1e-8)
                )
            )
        except Exception as e:
            logger.error(f"SMAPE calculation failed: {str(e)}")
            raise

    def _evaluate_baseline(self):

        try:
            logger.info("")
            logger.info("STEP 1 - BASELINE EVALUATION")
            logger.info("-" * 50)
            
            X , y = self._prepare_data(self.test_df)
            baseline_pred = X[self.config.baseline_feature]

            baseline_mae  = float(mean_absolute_error(y, baseline_pred))

            logger.info(f"Baseline MAE : {baseline_mae:.6f}")
            logger.info("PASSED - Baseline evaluated")

            return baseline_mae

        except Exception as e:
            logger.error(f"Baseline evaluation failed: {str(e)}")
            raise

            
        except Exception as e:
            raise

    
    def _deployment_decision(self , baseline_mae=0.020326390667323318):

        try:
            logger.info("")
            logger.info("STEP 3 - DEPLOYMENT DECISION")
            logger.info("-" * 50)
            
            passing = {}
            model_eval_files = glob.glob(os.path.join(self.config.model_eval_results , "*json"))
            model_result =  max(model_eval_files ,key=os.path.getmtime)
            
            with open(model_result ,'r') as f:
                data = json.load(f)
                #return data
            
            for model_name , metrics in data.items():
                test_mae = metrics['metrics']['test_mae']
                improvement = (baseline_mae - test_mae) / baseline_mae * 100

                logger.info(f"Evaluating {model_name}: MAE={test_mae:.6f}, Improvement={improvement:.2f}")

                logger.info(f"Baseline MAE  : {baseline_mae:.6f}")
                logger.info(f"Model MAE     : {test_mae:.6f}")
                logger.info(f"Improvement   : {improvement:.2f}%")
                logger.info(f"Threshold     : {self.config.improvement_threshold * 100:.0f}%")

                if improvement >= self.config.improvement_threshold * 100:
                    passing[model_name] = improvement

            if not passing:
                logger.error("FAILED - No model beat the baseline threshold")
                logger.error("Nothing deployed; current champion stays in production")
                return False, None, None
            
            best_model_name = max(passing ,key=passing.get)
            logger.info(f"PASSED - {best_model_name} beats threshold (improvement: {passing[best_model_name]:.2f}%)")
            return True, best_model_name, passing[best_model_name]
                
        except Exception as e:
            raise

    def log_to_mlflow(self):
        try:
            logger.info("")
            logger.info("STEP 2 - EVALUATE + LOG TO MLFLOW")
            logger.info("-" * 50)
            self.model = self._latest_models()

            X_test, y_test = self._prepare_data(self.test_df)
            X_val, y_val = self._prepare_data(self.val_df)

            mlflow.end_run() 
            mlflow.set_tracking_uri(self.config.mlflow_tracking_uri)
            mlflow.set_experiment(self.config.mlflow_experiment)

            report = {}
            for model_path in self.model:
                #print(m)
                model_name = os.path.basename(model_path).replace(".pkl", "")
                
                with open(model_path , 'rb') as f:
                    model = pickle.load(f)
                
                val_pred = model.predict(X_val)
                val_mae = float(mean_absolute_error(y_val, val_pred))
                val_smape = self._smape(y_val.values, val_pred)

                test_pred = model.predict(X_test)
                test_mae = float(mean_absolute_error(y_test, test_pred))
                test_smape = self._smape(y_test.values, test_pred)

                metrics = {
                        "val_mae": float(mean_absolute_error(y_val, val_pred)),
                        "val_smape": float(self._smape(y_val.values, val_pred)),
                        "test_mae": float(mean_absolute_error(y_test, test_pred)),
                        "test_smape": float(self._smape(y_test.values, test_pred))
                    }
                
        
                with mlflow.start_run(run_name=model_name):

                    params = self.config.model_training[model_name]['params']
                    mlflow.log_params(params)
                    mlflow.log_metrics(metrics)
                    mlflow.set_tag("model_type", model_name)
                    mlflow.set_tag("evaluated_at", datetime.now().isoformat())

                    log_fn = get_mlflow_logger(model_name)
                    log_fn(model, artifact_path="model")

                    report[model_name] = {
                            "params": params,
                            "metrics": metrics,
                            "timestamp": datetime.now().isoformat()
                        }
                    logger.info(f"{model_name} -> val_mae={val_mae:.6f}, test_mae={test_mae:.6f}")
                    logger.info("PASSED - logged to MLflow")
                    logger.info("-" * 50)

            timestamp = datetime.now().strftime("%Y_%m_%d_%H")
            report_path = os.path.join(self.config.model_eval_results, f"Model_evaluation_report_{timestamp}.json")
            with open(report_path, "w") as f:
                    json.dump(report, f, indent=4)
                        
                        # Log report as an artifact in MLflow
            #mlflow.log_artifact(report_path)
                        
            logger.info(f"Report saved {report_path}")
            logger.info("PASSED - MLflow logging and report generation complete")
            return report

        except Exception as e:
                logger.error(f"Failed to log to MLflow: {str(e)}")
                raise
    
    def run(self):
        try:
            logger.info("=" * 50)
            logger.info("MODEL EVALUATION PIPELINE STARTED")
            logger.info("=" * 50)

            baseline_mae = self._evaluate_baseline()
            self.log_to_mlflow()
            promoted, best_model_name, improvement = self._deployment_decision(baseline_mae)

            logger.info("=" * 50)
            logger.info(f"Best model : {best_model_name if promoted else 'None'}")
            logger.info("MODEL EVALUATION PIPELINE COMPLETE")
            logger.info("=" * 50)
            return promoted, best_model_name, improvement

        except Exception as e:
            logger.error(f"Model evaluation pipeline failed: {str(e)}")
            raise
        finally:
            self.test_df = None
            self.val_df = None
            gc.collect()
            logger.info("Memory cleared")




            
    

            


In [44]:
xc = Config_manager()
xc = xc.get_model_evaluation_config()
xc = Model_evalulation(xc)
xc._read_data("val")

[2026-07-12 18:53:05,157: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-12 18:53:05,160: INFO: common: Directory created (or already exists) at: artifacts]
[2026-07-12 18:53:05,162: INFO: common: Directory created (or already exists) at: artifacts/Model_evaluation/]
[2026-07-12 18:53:05,166: INFO: common: Directory created (or already exists) at: artifacts/Model_evaluation/evaluation_results/]


{'ResponseMetadata': {'RequestId': '6C3N8XBDMR523606', 'HostId': 'D4+H6l+IB6cCz8KACdQoscs8anPA3CFYFh37Fr18Nn75h9LadnHkg4RbdOJrMccYnE9puD0D40o=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'D4+H6l+IB6cCz8KACdQoscs8anPA3CFYFh37Fr18Nn75h9LadnHkg4RbdOJrMccYnE9puD0D40o=', 'x-amz-request-id': '6C3N8XBDMR523606', 'date': 'Sun, 12 Jul 2026 13:23:10 GMT', 'x-amz-bucket-region': 'us-east-1', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'feature_engineering/', 'LastModified': datetime.datetime(2026, 7, 11, 8, 3, 47, tzinfo=tzutc()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'feature_engineering/run__2026_07_12_15/test__2026_07_12_15/test_data.csv', 'LastModified': datetime.datetime(2026, 7, 12, 10, 20, 40, tzinfo=tzutc()), 'ETag': '"4475aee701156d287f0191d22c2bb6

,lag_1,lag_2,lag_3,lag_4,lag_96,rolling_std_4,bot_score
0,0.730340,0.734904,0.749471,0.753426,0.700161,0.034026,0.672171
1,0.672171,0.730340,0.734904,0.749471,0.711137,0.031365,0.686531
2,0.686531,0.672171,0.730340,0.734904,0.739297,0.034384,0.744104
3,0.744104,0.686531,0.672171,0.730340,0.736985,0.035150,0.733760
4,0.733760,0.744104,0.686531,0.672171,0.779504,0.030037,0.754582
...,...,...,...,...,...,...,...
4058,0.961362,0.962397,0.971314,0.976024,0.895895,0.006093,0.956739
4059,0.956739,0.961362,0.962397,0.971314,0.919774,0.016923,0.926680
4060,0.926680,0.956739,0.961362,0.962397,0.918614,0.043598,0.866668
4061,0.866668,0.926680,0.956739,0.961362,0.924156,0.037729,0.907414


In [44]:
import mlflow
# This will print the actual local folder where MLflow is dumping your models
print(f"Artifact URI: {mlflow.get_artifact_uri()}")

2026/06/25 16:19:57 WARNING mlflow.tracking.fluent: No active run found. A new active run will be created. If this is not intended, please create a run using `mlflow.start_run()` first.


Artifact URI: file:///d:/PredictBot-Score-MLOps/mlruns/1/d3208b712b5945c387e011d6949c5d4a/artifacts
